In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch

from google.colab import drive
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

In [2]:
import csv, os

os.makedirs("/content/results", exist_ok=True)
LOG = "/content/results/runs.csv"

def log_run(model_name, seed, lr, epochs, eff_batch, total_params, trainable_params, best_dev_acc):
    new = not os.path.exists(LOG)
    with open(LOG, "a", newline="") as f:
        w = csv.writer(f)
        if new:
            w.writerow(["model", "seed", "lr", "epochs", "eff_batch",
                        "total_params", "trainable_params", "best_dev_acc"])
        w.writerow([model_name, seed, lr, epochs, eff_batch,
                    total_params, trainable_params, round(best_dev_acc * 100, 2)])

In [3]:
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

SEED = 2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

PyTorch version: 2.14.1+cu130
GPU available: True
GPU: Tesla T4
Device: cuda


In [4]:
import pandas as pd

file_path = '/content/train.jsonl'
df = pd.read_json(file_path,lines=True)

file_path = '/content/val.jsonl'
val_df = pd.read_json(file_path,lines=True)

val_df.head()


,premise,hypothesis,label,idx
0,"Dana Reeve, the widow of the actor Christopher...",Christopher Reeve had an accident.,not_entailment,0
1,"Yet, we now are discovering that antibiotics a...",Bacteria is winning the war against antibiotics.,entailment,1
2,Cairo is now home to some 15 million people - ...,15 million tonnes of rubbish are produced dail...,not_entailment,2
3,"The Amish community in Pennsylvania, which num...",Pennsylvania has the biggest Amish community i...,not_entailment,3
4,Security forces were on high alert after an el...,Security forces were on high alert after a cam...,entailment,4


In [5]:
label_mapping = {
    "entailment": 0,
    "not_entailment": 1
}

id2label = {
    0: "entailment",
    1: "not_entailment"
}

df["label_id"] = df["label"].map(label_mapping)
val_df["label_id"] = val_df["label"].map(label_mapping)

print(df["label_id"].value_counts())
print(val_df["label_id"].value_counts())

label_id
0    1249
1    1241
Name: count, dtype: int64
label_id
0    146
1    131
Name: count, dtype: int64


In [6]:
from sklearn.model_selection import train_test_split
train_df, holdout_df = train_test_split(
    df, test_size=0.1, random_state=42, stratify=df["label_id"]
)

In [7]:
print(train_df['label'].unique())

['entailment' 'not_entailment']


In [8]:
assert train_df["label_id"].notna().all()
assert holdout_df["label_id"].notna().all()
assert val_df["label_id"].notna().all()

In [9]:
MODEL_NAME = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    torch_dtype=torch.float32,
).to(device)

# model.gradient_checkpointing_enable()   # new
print(model.dtype)

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  371MB            

... (2 outputs left out)

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier

model.safetensors: reconstructing file:   0%|          |  0.00B /  371MB            

model.safetensors: downloading bytes:           |  0.00B            

torch.float32


In [10]:
from transformers import DataCollatorWithPadding

class RTEDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=128):
        self.premises = dataframe["premise"].astype(str).tolist()
        self.hypotheses = dataframe["hypothesis"].astype(str).tolist()
        self.labels = dataframe["label_id"].astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        encoding = self.tokenizer(
            self.premises[index],
            self.hypotheses[index],
            truncation=True,
            max_length=self.max_length,      # removed padding="max_length"
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in encoding.items()
                if k != "token_type_ids"}    # drop token_type_ids
        item["labels"] = torch.tensor(self.labels[index], dtype=torch.long)
        return item

MAX_LENGTH = 128
BATCH_SIZE = 4                               # was 16
ACCUM_STEPS = 4                              # effective batch = 16

collator = DataCollatorWithPadding(tokenizer)   # pads per batch

train_dataset = RTEDataset(train_df, tokenizer, MAX_LENGTH)
holdout_dataset = RTEDataset(holdout_df, tokenizer, MAX_LENGTH)
val_dataset = RTEDataset(val_df, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator)
holdout_loader = DataLoader(holdout_dataset, batch_size=16, shuffle=False, collate_fn=collator)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=collator)

In [11]:
import math, gc
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW

gc.collect()
torch.cuda.empty_cache()

EPOCHS = 5          # you had 1; use 1 only as a smoke test
LR = 2e-5              # lower than 2e-5 for large
WARMUP = 0.1
USE_FP16 = True        # set False if you get NaN again

no_decay = ["bias", "LayerNorm.weight"]
params = [
    {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], "weight_decay": 0.01},
    {"params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], "weight_decay": 0.0},
]
optimizer = AdamW(params, lr=LR)

steps_per_epoch = math.ceil(len(train_loader) / ACCUM_STEPS)   # changed
total_steps = EPOCHS * steps_per_epoch
scheduler = get_linear_schedule_with_warmup(
    optimizer, int(WARMUP * total_steps), total_steps
)

scaler = torch.amp.GradScaler("cuda", enabled=USE_FP16)

def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for b in loader:
            b = {k: v.to(device) for k, v in b.items()}
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_FP16):
                out = model(**b)
            preds += out.logits.argmax(-1).cpu().tolist()
            labels += b["labels"].cpu().tolist()
    return accuracy_score(labels, preds), preds, labels

best_acc, best_state = 0, None
for epoch in range(EPOCHS):
    model.train()
    total = 0
    optimizer.zero_grad()

    for step, b in enumerate(train_loader):
        b = {k: v.to(device) for k, v in b.items()}

        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_FP16):
            loss = model(**b).loss / ACCUM_STEPS

        scaler.scale(loss).backward()
        total += loss.item() * ACCUM_STEPS

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

    hold_acc, _, _ = evaluate(model, holdout_loader)
    print(f"Epoch {epoch+1} | loss {total/len(train_loader):.4f} | holdout acc {hold_acc*100:.2f}")

    if hold_acc > best_acc:
        best_acc = hold_acc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}  # CPU copy

model.load_state_dict(best_state)
print("Best holdout acc:", best_acc)

val_acc, preds, labels = evaluate(model, val_loader)
print(f"Official dev acc: {val_acc*100:.2f}")
os.makedirs("/content/results/preds", exist_ok=True)
name = MODEL_NAME.split("/")[-1]
pd.DataFrame({
    "id": range(len(preds)),
    "prediction": [id2label[p] for p in preds],
    "label": [id2label[l] for l in labels],
}).to_csv(f"/content/results/preds/{name}_seed{SEED}.csv", index=False)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total:,} | Trainable: {trainable:,}")

log_run(MODEL_NAME, SEED, LR, EPOCHS, BATCH_SIZE * ACCUM_STEPS, total, trainable, val_acc)

Epoch 1 | loss 0.6891 | holdout acc 71.49
Epoch 2 | loss 0.4459 | holdout acc 81.93
Epoch 3 | loss 0.2224 | holdout acc 82.73
Epoch 4 | loss 0.0818 | holdout acc 81.12
Epoch 5 | loss 0.0421 | holdout acc 81.93
Best holdout acc: 0.8273092369477911
Official dev acc: 83.39
Total params: 184,423,682 | Trainable: 184,423,682
